# Vibrational modes of a water molecule

This notebook constructs an isolated H₂O molecule, relaxes its geometry with GPAW, builds a finite-difference Hessian with ASE's `Vibrations` class, reports the harmonic frequencies and zero-point energy, and animates one normal mode.

### Learning goals

- connect force calculations at displaced geometries to the Hessian and normal modes;
- distinguish molecular translations and rotations from internal vibrations;
- interpret real and imaginary frequencies without overdiagnosing a transition state;
- understand the numerical settings that control finite-difference frequencies; and
- display an ASE mode trajectory as an inline notebook animation.

Distances are in ångström (Å), energies in electronvolts (eV), forces in eV/Å, and vibrational energies and wavenumbers in millielectronvolts (meV) and inverse centimetres (cm⁻¹), respectively.

## Constructing and boxing the molecule

The first `molecule('H2O', vacuum=3.5)` object is immediately replaced by the manually constructed `Atoms` object, so it does not affect the calculation. In the replacement, the symbol string `'H2O'` orders the atoms as H, H, O, whereas the coordinates place atom 0 at the vertex of the specified 104.51° angle. The angle encoded by these starting coordinates is therefore H–H–O rather than H–O–H. Because ASE does not store chemical bonds, the optimizer can still rearrange these nuclei into a water minimum, but `Atoms('OH2', ...)` with O at the vertex would express the intended starting geometry more clearly. This code is left unchanged in this Markdown-only pass.

`center(vacuum=3.5)` creates a finite, non-periodic cell with at least 3.5 Å between the molecule and each cell face. A finite cell is required by the real-space electronic-structure calculation even though the molecule itself is not periodic.

In [1]:
from math import cos, pi, sin

from gpaw import GPAW

from ase import Atoms
from ase.build import molecule
from ase.optimize import QuasiNewton
from ase.vibrations import Vibrations

# Water molecule:
h2o = molecule('H2O', vacuum=3.5)
d = 0.9575
t = pi / 180 * 104.51

h2o = Atoms(
    'H2O', positions=[(0, 0, 0), (d, 0, 0), (d * cos(t), d * sin(t), 0)]
)

h2o.center(vacuum=3.5)

## Relaxing the molecular geometry

GPAW supplies energies and forces using localized-orbital (`lcao`) mode with a double-zeta polarized (`dzp`) basis. `symmetry='off'` prevents symmetry operations from obscuring the deliberately displaced geometries used later. Calculator details and self-consistent-field progress are written to `h2o.txt`.

`QuasiNewton` relaxes the atomic positions until the maximum force falls below 0.05 eV/Å. The stored output reaches 0.0422 eV/Å after seven optimizer steps. This is adequate for demonstrating the workflow, but vibrational frequencies—especially the nominally zero translations and rotations—are more sensitive to residual forces than an ordinary geometry illustration.

The final mode trajectories imply two O–H distances of approximately 0.985 Å and an H–O–H angle of approximately 102.87° for this relaxed structure. These are results of the selected LCAO/DZP setup and force threshold, not converged reference values.

## Building the finite-difference Hessian

For $N$ atoms, `Vibrations` evaluates the Cartesian force response to small positive and negative displacements. With the defaults `delta=0.01 Å` and `nfree=2`, a Hessian element is approximated by a central difference,

$$
H_{a\alpha,b\beta} \approx -\frac{F_{b\beta}(R + \delta e_{a\alpha}) - F_{b\beta}(R - \delta e_{a\alpha})}{2\delta}.
$$

Water has nine Cartesian degrees of freedom, so the run requires one equilibrium force evaluation plus two displacements for each degree of freedom: $1 + 2 \times 9 = 19$ force calculations. Results are cached under `files/advanced/tutorial_09/vib/`; rerunning `vib.run()` reuses complete cache entries instead of repeating those calculations.

ASE mass-weights the Hessian,

$$
D_{a\alpha,b\beta} = \frac{H_{a\alpha,b\beta}}{\sqrt{m_a m_b}},
$$

and diagonalizes $D$. Its eigenvectors are the normal modes and its eigenvalues are proportional to $\omega_k^2$. The `frederiksen` option corrects each displaced force set for residual net force before assembling the Hessian; it improves translational consistency but does not automatically project out every translational and rotational mode.

In [2]:
h2o.calc = GPAW(txt='../files/advanced/tutorial_09/h2o.txt', mode='lcao', basis='dzp', symmetry='off')

QuasiNewton(h2o).run(fmax=0.05)


"""Calculate the vibrational modes of a H2O molecule."""

# Create vibration calculator
vib = Vibrations(h2o, name='../files/advanced/tutorial_09/vib')
vib.run()
vib.summary(method='frederiksen')

# Make trajectory files to visualize normal modes:
for mode in range(9):
    vib.write_mode(mode)

                Step[ FC]     Time          Energy          fmax
BFGSLineSearch:    0[  0] 20:59:37       -9.313801       5.8617
BFGSLineSearch:    1[  1] 20:59:38      -10.845761       7.7626
BFGSLineSearch:    2[  2] 20:59:38      -12.601153       7.3396
BFGSLineSearch:    3[  3] 20:59:38      -13.334464       4.4366
BFGSLineSearch:    4[  5] 20:59:39      -13.506025       2.8879
BFGSLineSearch:    5[  7] 20:59:40      -13.704737       1.0861
BFGSLineSearch:    6[  9] 20:59:41      -13.719590       0.1243
BFGSLineSearch:    7[ 10] 20:59:41      -13.719895       0.0422
---------------------
  #    meV     cm^-1
---------------------
  0   23.0i    185.4i
  1   20.9i    168.6i
  2   17.2i    138.5i
  3    0.9       7.3
  4    2.5      20.0
  5    5.9      47.7
  6  190.7    1538.4
  7  450.2    3631.3
  8  464.8    3748.6
---------------------
Zero-point energy: 0.558 eV


## Interpreting the vibrational summary

A nonlinear three-atom molecule has $3N=9$ Cartesian normal modes. Six describe whole-molecule translation and rotation and should approach zero frequency; the remaining three are internal vibrations. The physically meaningful high-frequency modes in the stored output are:

| Mode | Energy (meV) | Wavenumber (cm⁻¹) | Interpretation |
|---:|---:|---:|---|
| 6 | 190.7 | 1538.4 | H–O–H bending |
| 7 | 450.2 | 3631.3 | O–H stretching |
| 8 | 464.8 | 3748.6 | O–H stretching |

The first six values, from 185.4i to 47.7 cm⁻¹, belong to the translation/rotation subspace rather than six additional molecular vibrations. The suffix `i` means the corresponding Hessian eigenvalue is negative, $\omega_k^2 < 0$. For a genuinely optimized minimum, a substantial imaginary **internal** mode can indicate an instability; here, however, these low modes are external motions contaminated by residual forces and numerical breaking of exact translational and rotational invariance. Their magnitudes are larger than ideal and should be treated as a convergence diagnostic, not silently interpreted as physical vibrations.

ASE reports a harmonic zero-point energy of 0.558 eV from the positive real mode energies it includes:

$$
E_{ZPE} = \frac{1}{2}\sum_k \hbar\omega_k.
$$

The value inherits the geometry, calculator, finite-difference step, and treatment of the low-frequency modes. It should not be presented as a converged molecular benchmark.

## Writing and animating a normal mode

`vib.write_mode(mode)` writes 30 geometries by default for each of the nine harmonic eigenvectors. These frames are a sinusoidal visualization of a mass-weighted normal mode, not a molecular-dynamics trajectory and not a real-time prediction. The animation cell reads mode 8, the highest-frequency O–H stretching mode, and omits the final repeated frame when constructing the loop.

In [3]:
import matplotlib.animation as animation
import matplotlib.pyplot as plt

from ase.io import read
from ase.visualize.plot import plot_atoms
from IPython.display import HTML

configs = read('../files/advanced/tutorial_09/vib.8.traj', ':')

fig, ax = plt.subplots()

def animate(i):
    # Remove the previous atomic plot
    [p.remove() for p in ax.patches]
    plot_atoms(configs[i], ax, rotation=('0x,0y,0z'), show_unit_cell=1)
    ax.set_xlim(-4, 7)
    ax.set_ylim(-4, 7)
    ax.set_axis_off()
    return (ax,)

ani = animation.FuncAnimation(
    fig,
    animate,
    repeat=True,
    frames=len(configs) - 1,
    interval=200,
)

html = HTML(ani.to_jshtml())
plt.close(fig)  # Prevent an extra static figure
html

### Inline animation output

`FuncAnimation` creates the animation object, while `HTML(ani.to_jshtml())` serializes its frames and controls into notebook-compatible JavaScript. Retaining the object as `ani` prevents it from being garbage-collected before rendering, and closing the Matplotlib figure suppresses an extra static copy.

> **Fresh-kernel caution:** the current cell uses `fig` and `ax` without creating them. Its stored HTML output was produced in a kernel where those variables already existed. For reproducible execution from a fresh kernel, add `fig, ax = plt.subplots()` before defining or calling `animate`. The code is preserved here because this request changes Markdown only.

## Takeaways and possible extensions

Finite-difference vibrational analysis converts force responses into a mass-weighted Hessian, whose eigenvectors describe normal-mode displacement patterns and whose eigenvalues determine harmonic frequencies. For isolated nonlinear H₂O, the central interpretation is six near-zero external motions plus three internal modes: one bend and two O–H stretches.

The present calculation is tutorial-scale. It uses the harmonic approximation, a 0.05 eV/Å geometry threshold, a single 0.01 Å displacement, an LCAO/DZP electronic-structure model, finite vacuum, and no systematic basis, grid, cell, or force-convergence study. It also reports frequencies but not infrared intensities or anharmonic corrections.

The most valuable next addition would be a **mode-classification and convergence cell** that reports the optimized O–H distances and angle, labels the six external and three internal modes, and repeats the calculation with a tighter geometry threshold and several displacement sizes. The appreciable imaginary external modes make that diagnostic more important than simply adding another animation. I would follow it with:

1. a corrected, unambiguous initial construction using O at the H–O–H vertex, plus a saved optimized geometry;
2. a force threshold near 0.005 eV/Å and a displacement study such as 0.005, 0.01, and 0.02 Å;
3. comparison of `standard` and `frederiksen` force treatments and checks of basis, grid, and vacuum sensitivity;
4. a compact frequency table or CSV with mode index, energy, wavenumber, real/imaginary character, and assignment;
5. infrared intensities using ASE's `Infrared` workflow, followed by a broadened stick spectrum;
6. an H₂O/D₂O isotope comparison to demonstrate the approximate mass dependence $\omega \propto 1/\sqrt{\mu}$; and
7. harmonic thermochemistry only after the low-frequency modes and numerical convergence have been handled explicitly.

A self-contained animation cell with a mode selector and frequency label would improve presentation, but the convergence and mode-assignment checks should come first scientifically.